# 04. Exportar associações únicas

Mesma regra do [`03_avaliar.ipynb`](03_avaliar.ipynb): melhor CPF por Censo
entre pares com `p ≥ THRESHOLD_AVALIACAO`; grava só os CPFs que ficaram com
**um** Censo. Sem cluster, sem greedy, sem métricas de ouro — avaliação fica no 03.

Saída: `splink_atribuicao.parquet`.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_PREDICTIONS,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    print_paths,
    require_input,
)

T = THRESHOLD_AVALIACAO

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
con = get_connection()
drop_splink_temp_tables(con)
print('T:', T)


In [ ]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
WHERE match_probability >= {T}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute('''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT m.*
FROM melhor_por_censo m
JOIN (
    SELECT unique_id_cpf
    FROM melhor_por_censo
    GROUP BY 1
    HAVING COUNT(*) = 1
) c ON c.unique_id_cpf = m.unique_id_cpf
''')

n_melhor = con.execute('SELECT COUNT(*) FROM melhor_por_censo').fetchone()[0]
n_unicas = con.execute('SELECT COUNT(*) FROM associacoes_unicas').fetchone()[0]
print('Censos com par >= T:', f'{n_melhor:,}')
print('Associações únicas:', f'{n_unicas:,}')


In [ ]:
export_parquet(con, 'associacoes_unicas', path=SPLINK_ATRIBUICAO)
print('Exportado:', SPLINK_ATRIBUICAO)
con.close()
